# 📖 Notebook 1: Data Classification

Before you can protect personal data, you need to **find** it. In this notebook, we'll scan a database to discover PII (Personally Identifiable Information), classify each column by sensitivity level, and build a classification registry.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to scan database schemas to detect PII columns
- The four classification levels: Public, Internal, Confidential, Restricted
- How to build automated PII detection using pattern matching
- How to maintain a data classification registry
- Why unclassified data is treated as Restricted by default

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/privacy-review
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `privacy_review`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import re
from datetime import datetime

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 🔍 Step 1: Discover All Tables and Columns

The first step in any privacy review is understanding **what data you have**. Most engineers are surprised by how much PII is scattered across their database.

We'll query PostgreSQL's `information_schema` to get every table and column in our database. This is the same approach real privacy scanners use.

In [ ]:
def discover_schema():
    """Query the database schema to find all tables and columns."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT table_name, column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'public'
        ORDER BY table_name, ordinal_position
    """)

    schema = {}
    for table, column, dtype, nullable in cursor.fetchall():
        if table not in schema:
            schema[table] = []
        schema[table].append({
            "column": column,
            "type": dtype,
            "nullable": nullable == "YES"
        })

    conn.close()
    return schema

schema = discover_schema()

print("📋 Database Schema Discovery")
print("=" * 60)
for table, columns in schema.items():
    print(f"\n📁 {table} ({len(columns)} columns)")
    for col in columns:
        print(f"   ├── {col['column']:<30} {col['type']}")

## 🏷️ Step 2: Automated PII Detection

Now we need to figure out which of these columns contain personal data. We'll build a **PII scanner** that uses two approaches:

1. **Column name pattern matching** — column names like `email`, `ssn`, `phone` are obvious PII
2. **Data content sampling** — check actual values for patterns like email addresses or phone numbers

Real tools like Microsoft Purview, AWS Macie, or Google DLP use machine learning for this. Pattern matching is cheap and catches the structured cases — Step 3b measures exactly how much of the rest it misses, and the answer is sobering.

In [ ]:
# PII detection rules based on column names
# Each rule maps a pattern to a classification level and PII type.
#
# Patterns match whole `_`-separated *tokens* of a column name, never raw
# substrings. Substring matching is the classic way to wreck a scanner:
# "id" is a substring of "national_id" (so every primary key would be tagged
# as a government ID), and "ip" is a substring of "description" and
# "shipped_at". Both mistakes are silent — you get a registry full of
# confident nonsense.

PII_COLUMN_PATTERNS = {
    # RESTRICTED — highest sensitivity
    "restricted": {
        "government_id": ["ssn", "social_security", "tax_id", "national_id", "passport"],
        "financial":     ["card_number", "account_number", "routing_number", "credit_card"],
        "health":        ["diagnosis", "medical_record", "health_condition", "prescription"],
        "credentials":   ["password", "password_hash", "secret", "token"],
        # Exact date of birth + ZIP + sex re-identifies most of the US
        # population (Sweeney, 2000), which is why db/init.sql also files
        # users.date_of_birth as RESTRICTED. Scanner and registry must agree
        # on the taxonomy or the registry becomes a coin flip.
        "dob":           ["date_of_birth", "birth_date", "dob", "birthday"],
    },
    # CONFIDENTIAL — identifies a person
    "confidential": {
        "name":     ["first_name", "last_name", "full_name", "display_name", "shipping_name"],
        "email":    ["email", "email_address", "contact_email"],
        "phone":    ["phone", "phone_number", "mobile", "telephone"],
        "address":  ["street_address", "address", "shipping_address", "billing_address"],
        "location": ["zip", "zip_code", "postal_code", "geo_city", "geo_lat", "geo_lon",
                     "latitude", "longitude"],
        "network":  ["ip_address", "ip", "mac_address"],
    },
    # INTERNAL — company use only
    "internal": {
        "location":        ["city", "state", "country", "region", "geo_country"],
        "device":          ["user_agent", "device_type", "browser"],
        "metadata":        ["account_status", "signup_source", "created_at", "updated_at"],
        # A pseudonymous key is still personal data under GDPR Recital 26: it
        # singles a person out even though it carries no name.
        "pseudonymous_id": ["user_id", "customer_id", "account_id", "device_id", "session_id"],
    }
}

# Ordered least → most sensitive. Used everywhere we compare two levels.
SEVERITY = ["public", "internal", "confidential", "restricted"]

def _pattern_matches(pattern, column):
    """True when `pattern` lines up with whole `_`-separated tokens of `column`."""
    p, c = pattern.split("_"), column.split("_")
    return any(c[i:i + len(p)] == p for i in range(len(c) - len(p) + 1))

def classify_column_by_name(column_name):
    """Classify a column from its name.

    The most severe matching level wins. Inside a level the most *specific*
    (longest) pattern wins, so `ip_address` is tagged `network` and not the
    shorter `address` match.
    """
    col = column_name.lower()

    for level in ("restricted", "confidential", "internal"):
        hits = [(len(pattern.split("_")), pii_type)
                for pii_type, patterns in PII_COLUMN_PATTERNS[level].items()
                for pattern in patterns
                if _pattern_matches(pattern, col)]
        if hits:
            return level, max(hits)[1]

    return "public", None

# Test our classifier on a few examples — including the ones that a naive
# substring scanner gets wrong, and one it simply cannot see.
test_columns = ["email", "ssn", "first_name", "ip_address", "date_of_birth",
                "product_id", "price", "user_agent", "user_id",
                "id", "description", "shipped_at", "card_last_four"]

print("🧪 Column Name Classification Test")
print("=" * 55)
for col in test_columns:
    level, pii_type = classify_column_by_name(col)
    icons = {"restricted": "🔴", "confidential": "🟡", "internal": "🔵", "public": "🟢"}
    pii_label = f" ({pii_type})" if pii_type else ""
    print(f"  {icons[level]} {col:<25} → {level}{pii_label}")

# Regression guards for the substring trap. `id` and `shipped_at` carry no PII;
# a scanner that flags them floods the privacy team with noise, and a scanner
# that mis-types `ip_address` sends the wrong retention rule to the wrong data.
assert classify_column_by_name("id") == ("public", None), "primary keys are not government IDs"
assert classify_column_by_name("shipped_at") == ("public", None), "'ip' is not a token of 'shipped_at'"
assert classify_column_by_name("ip_address") == ("confidential", "network"), "specific pattern must win"
assert classify_column_by_name("date_of_birth")[0] == "restricted", "must match db/init.sql registry"
print("\n✅ Classifier regression checks passed")
print("⚠️  Note `card_last_four` → public: the name scanner has no pattern for it.")
print("   A human classified it as CONFIDENTIAL in db/init.sql. Step 5 shows why")
print("   the scanner must never overwrite a human's more severe judgement.")


## 📊 Step 3: Content-Based PII Scanning

Column names don't always tell the full story. A column called `description` might contain PII that users typed in (like "My email is john@gmail.com"). We need to **scan the actual data** too.

This is especially important for **free-text fields** like support tickets, comments, and notes.

In [ ]:
# Content-based PII detection using regular expressions
# These patterns match common PII formats in text

PII_CONTENT_PATTERNS = {
    "email": {
        "regex": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        "classification": "confidential",
        "description": "Email address found in text"
    },
    "ssn": {
        "regex": r'\b\d{3}-\d{2}-\d{4}\b',
        "classification": "restricted",
        "description": "SSN pattern (XXX-XX-XXXX) found in text"
    },
    "phone_us": {
        "regex": r'\b(?:\+?1[-.]?)?\(?\d{3}\)?[-.]?\d{3}[-.]?\d{4}\b',
        "classification": "confidential",
        "description": "US phone number found in text"
    },
    "credit_card": {
        "regex": r'\b(?:4\d{3}|5[1-5]\d{2}|6011|3[47]\d{2})[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b',
        "classification": "restricted",
        "description": "Credit card number found in text"
    },
    "dob": {
        "regex": r'\b(?:DOB|date of birth|born on|birthday)[:\s]+\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b',
        "classification": "restricted",
        "description": "Date of birth mentioned in text"
    },
    "ssn_last4": {
        "regex": r'(?:SSN|social security)\s*(?:last\s*4|ending)[:\s]*\d{4}',
        "classification": "restricted",
        "description": "SSN reference found in text"
    }
}

def scan_text_for_pii(text):
    """Scan a text string for PII patterns. Returns list of findings."""
    if not text:
        return []

    findings = []
    for pii_type, config in PII_CONTENT_PATTERNS.items():
        matches = re.findall(config["regex"], text, re.IGNORECASE)
        if matches:
            findings.append({
                "type": pii_type,
                "classification": config["classification"],
                "matches": matches,
                "description": config["description"]
            })
    return findings

# Test on our support ticket data — this is where PII often hides
print("🔎 Scanning Support Tickets for Hidden PII")
print("=" * 60)

conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT id, subject, description, internal_notes FROM support_tickets")

for ticket_id, subject, description, notes in cursor.fetchall():
    print(f"\n📝 Ticket #{ticket_id}: {subject}")

    # Scan the description field
    desc_findings = scan_text_for_pii(description)
    if desc_findings:
        print(f"   ⚠️  PII in description:")
        for f in desc_findings:
            icon = "🔴" if f["classification"] == "restricted" else "🟡"
            print(f"      {icon} {f['description']} — found: {f['matches']}")

    # Scan internal notes
    notes_findings = scan_text_for_pii(notes)
    if notes_findings:
        print(f"   ⚠️  PII in internal_notes:")
        for f in notes_findings:
            icon = "🔴" if f["classification"] == "restricted" else "🟡"
            print(f"      {icon} {f['description']} — found: {f['matches']}")

    if not desc_findings and not notes_findings:
        # Deliberately NOT "✅ no PII". The scanner only knows the six patterns
        # above; silence means "nothing matched", which is a much weaker claim.
        print("   ➖ Nothing matched our patterns (this is NOT the same as 'no PII')")

conn.close()

## 📉 Step 3b: Measure What the Scanner *Misses*

The scan above printed a clean line for three of the five tickets. Two of those
three are not clean at all — one contains a full home address, another contains
a date of birth and a partial card number.

This is the most dangerous failure mode in privacy engineering: a tool that
reports "no findings" and gets read as "no PII". A regex scanner has no idea
what a person's name looks like, and its patterns are brittle — our `dob`
pattern wants `DOB: 03/15/1985` and quietly misses `DOB is 03/15/1985`.

So we measure it. Below is a **ground-truth set**: the same five ticket bodies,
with the PII a human reviewer marked in each. Running the scanner against it
gives us a recall number instead of a comfortable feeling.


In [ ]:
# Ground truth: what a *human* reviewer found in each seeded ticket.
# (Text copied from db/init.sql so this measurement never depends on whether
# notebook 4 has already redacted the rows.)
GROUND_TRUTH = [
    {
        "ticket": "Cannot reset password",
        "text": "Hi, my name is Alice Smith and I cannot reset my password. My email is "
                "alice.real@gmail.com and phone is 555-0101. Please help! "
                "[notes] Verified identity via SSN last 4: 1234. Reset link sent.",
        "pii": {"person_name", "email", "phone", "ssn_last4"},
    },
    {
        "ticket": "Wrong shipping address",
        "text": "Please update my address to 742 Evergreen Terrace, Springfield IL 62704. "
                "My order ORD-000005 shipped to wrong place. "
                "[notes] Customer provided new address. CC number ends in 4242.",
        "pii": {"street_address", "card_last4"},
    },
    {
        "ticket": "Billing question",
        "text": "I see a charge of $149.99 on my Visa ending 8901. My DOB is 03/15/1985 "
                "for verification. Can you explain this charge?",
        "pii": {"dob", "card_last4"},
    },
    {
        "ticket": "Account deletion request",
        "text": "I want to delete my account per GDPR. My full name is Tina Martin, "
                "email tina.m@gmail.com, SSN 456-78-9012.",
        "pii": {"person_name", "email", "ssn"},
    },
    {
        "ticket": "Refund request",
        "text": "Please refund order ORD-000008. The product was damaged. "
                "[notes] Refund approved. Customer Hank Davis, card ending 2345.",
        "pii": {"person_name", "card_last4"},
    },
]

# Our detector emits `phone_us`; the reviewer wrote `phone`. Normalise so the
# comparison is about substance, not spelling.
DETECTOR_ALIASES = {"phone_us": "phone", "credit_card": "card_number"}

print("📉 Detector Recall Against Human Ground Truth")
print("=" * 78)

found_total, expected_total = set(), set()
silent_but_dirty = 0

for i, case in enumerate(GROUND_TRUTH, 1):
    detected = {DETECTOR_ALIASES.get(f["type"], f["type"])
                for f in scan_text_for_pii(case["text"])}
    missed = case["pii"] - detected

    found_total |= {(i, p) for p in (case["pii"] & detected)}
    expected_total |= {(i, p) for p in case["pii"]}
    if not detected and case["pii"]:
        silent_but_dirty += 1

    status = "🟢" if not missed else ("🔴" if not detected else "🟡")
    print(f"\n  {status} Ticket #{i}: {case['ticket']}")
    print(f"     Human found:   {sorted(case['pii'])}")
    print(f"     Scanner found: {sorted(detected) or '— nothing —'}")
    if missed:
        print(f"     ❌ MISSED:      {sorted(missed)}")

recall = len(found_total) / len(expected_total)
print("\n" + "=" * 78)
print(f"  Recall: {len(found_total)}/{len(expected_total)} PII items = {recall:.0%}")
print(f"  Tickets where the scanner said nothing but PII was present: {silent_but_dirty}/{len(GROUND_TRUTH)}")

print("""
💡 Why these were missed:
   • person_name    — no regex recognises "Hank Davis"; needs an NER model.
   • street_address — free-form addresses have no reliable shape.
   • card_last4     — "card ending 2345" is four digits in a sentence;
                      a pattern loose enough to catch it would flag every
                      order number, price and ZIP code in the database.
   • dob            — the pattern wants "DOB: 03/15/1985" and the customer
                      wrote "DOB is 03/15/1985".

   The trade-off is real: every pattern you loosen to raise recall lowers
   precision, and a scanner that cries wolf gets switched off.
""")

# These assertions pin the *lesson*, not a lucky output.
assert recall < 1.0, (
    "This lab teaches that regex PII detection has false negatives. If you "
    "improved the patterns until recall hit 100%, add a harder example to "
    "GROUND_TRUTH — do not delete this assertion."
)
assert silent_but_dirty >= 1, (
    "At least one ticket must produce zero matches while containing PII — "
    "that is the exact situation where 'scan passed' gets misread as 'no PII'."
)
assert (1, "email") in found_total and (4, "ssn") in found_total, (
    "The patterns that DO work must keep working: a well-formed email and a "
    "well-formed SSN are the detector's whole job."
)
print("✅ Recall harness assertions passed — the scanner is honestly imperfect")


## 🗂️ Step 4: Full Database Classification Scan

Let's combine both approaches — column name matching AND content scanning — to classify every column in every table. This gives us a complete picture of where PII lives in our database.

This is what a real privacy scanner does when you first connect it to a database.

In [ ]:
def full_database_scan(sample_rows=50):
    """Scan the entire database and classify every column.

    Two signals per column: the column *name*, and a sample of the actual
    *values*. We run the content scan on every text column — not just the ones
    the name scanner shrugged at. A column called `email` can still hold an SSN
    a customer typed into the wrong box, and a column already tagged
    CONFIDENTIAL can still be hiding RESTRICTED content.
    """
    schema = discover_schema()
    results = []

    # Skip metadata/registry tables — we only want application tables
    skip_tables = {
        "data_classification_registry", "privacy_impact_assessments",
        "data_retention_policies", "purge_audit_log"
    }

    conn = get_db_connection()
    cursor = conn.cursor()

    for table, columns in schema.items():
        if table in skip_tables:
            continue

        for col_info in columns:
            col_name = col_info["column"]
            col_type = col_info["type"]

            # Step 1: classify by column name
            level, pii_type = classify_column_by_name(col_name)

            # Step 2: for text columns, also sample the content
            content_pii = []
            sampled = col_type in ("text", "character varying")
            if sampled:
                try:
                    cursor.execute(
                        f'SELECT "{col_name}" FROM "{table}" '
                        f'WHERE "{col_name}" IS NOT NULL LIMIT {int(sample_rows)}'
                    )
                    for (value,) in cursor.fetchall():
                        content_pii.extend(scan_text_for_pii(str(value)))
                except Exception as e:
                    # Never swallow this silently: a column we failed to sample
                    # is a column we have no content evidence for.
                    print(f"   ⚠️ Could not sample {table}.{col_name}: {e}")
                    sampled = False

            # Upgrade (never downgrade) the classification if the content is worse
            if content_pii:
                worst = max(content_pii, key=lambda f: SEVERITY.index(f["classification"]))
                if SEVERITY.index(worst["classification"]) > SEVERITY.index(level):
                    level = worst["classification"]
                    pii_type = "free_text"

            results.append({
                "table": table,
                "column": col_name,
                "type": col_type,
                "classification": level,
                "pii_type": pii_type,
                "content_findings": len(content_pii),
                "sampled": sampled,
            })

    conn.close()
    return results

# Run the full scan
scan_results = full_database_scan()

# Display results grouped by table
print("📊 Full Database Classification Report")
print("=" * 70)

icons = {"restricted": "🔴", "confidential": "🟡", "internal": "🔵", "public": "🟢"}
current_table = None

for r in scan_results:
    if r["table"] != current_table:
        current_table = r["table"]
        print(f"\n📁 {current_table}")

    pii_label = f" ({r['pii_type']})" if r["pii_type"] else ""
    content_flag = f" ⚠️ +{r['content_findings']} content matches" if r["content_findings"] else ""
    print(f"   {icons[r['classification']]} {r['column']:<30} {r['classification']}{pii_label}{content_flag}")

# Summary statistics
print("\n" + "=" * 70)
print("📈 Summary")
for level in ["restricted", "confidential", "internal", "public"]:
    count = sum(1 for r in scan_results if r["classification"] == level)
    print(f"   {icons[level]} {level.upper():<15} {count} columns")

not_sampled = [r for r in scan_results if not r["sampled"]]
print(f"\n   Content-scanned: {len(scan_results) - len(not_sampled)} text columns")
print(f"   Name-only:       {len(not_sampled)} non-text columns (numbers, dates, booleans)")
print("   ⚠️  'public' here means 'nothing matched', not 'proven safe' — see the")
print("      recall harness above.")

by_col = {(r["table"], r["column"]): r for r in scan_results}

# The scan must reproduce the two signals this notebook is about.
assert by_col[("users", "ssn")]["classification"] == "restricted", "name scan missed users.ssn"
assert by_col[("orders", "id")]["classification"] == "public", (
    "a primary key is not PII — if this fails the substring trap is back"
)
assert by_col[("support_tickets", "description")]["classification"] == "restricted", (
    "the content scan must upgrade support_tickets.description to RESTRICTED "
    "(ticket #4 contains a full SSN). If you already ran notebook 4, its retention "
    "purge may have redacted the ticket text — recreate the database with "
    "`docker compose down -v && docker compose up -d`."
)
assert any(r["pii_type"] == "free_text" for r in scan_results), (
    "at least one column must be classified by content, not by name — that is "
    "the whole point of scanning values"
)
print("\n✅ Scan assertions passed")


## 💾 Step 5: Save Classifications to the Registry

Once we've classified everything, we need to **persist** the results so other teams can look up the classification of any column. We'll save to both:

1. **PostgreSQL** — the `data_classification_registry` table (source of truth)
2. **Redis** — a cache for fast lookups during API requests

In a real system, this registry is queried every time someone writes a query or builds an API endpoint — "am I allowed to return this column to this caller?"

In [ ]:
def load_registry():
    """Current registry contents, keyed by (table, column)."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT table_name, column_name, classification, pii_type, classified_by
        FROM data_classification_registry
    """)
    rows = cursor.fetchall()
    conn.close()
    return {(t, c): {"classification": cl, "pii_type": pt, "classified_by": by}
            for t, c, cl, pt, by in rows}

def save_classifications_to_db(scan_results):
    """Merge scan results into the registry.

    The rule that matters: the scanner may INSERT a new row and may UPGRADE a
    row to a more severe level, but it must never DOWNGRADE one. A human filed
    `payment_methods.card_last_four` as CONFIDENTIAL; our name patterns have no
    rule for it and would cheerfully overwrite that with `public`. Blind
    upserts are how a registry quietly loses every judgement a human ever made.
    """
    existing = load_registry()
    conn = get_db_connection()
    cursor = conn.cursor()

    inserted, upgraded, kept, blocked = 0, 0, 0, []

    for r in scan_results:
        key = (r["table"], r["column"])
        prior = existing.get(key)

        # Columns the scanner thinks are clean are only interesting when the
        # registry disagrees — that is a downgrade attempt, and we log it.
        if r["classification"] == "public":
            if prior is not None:
                kept += 1
                blocked.append((key, "public", prior))
            continue

        if prior is None:
            cursor.execute("""
                INSERT INTO data_classification_registry
                    (table_name, column_name, classification, pii_type, notes, classified_by)
                VALUES (%s, %s, %s, %s, %s, %s)
                ON CONFLICT (table_name, column_name) DO NOTHING
            """, (r["table"], r["column"], r["classification"], r["pii_type"],
                  "Discovered by automated scan", "auto-scanner"))
            inserted += 1
            continue

        new_rank = SEVERITY.index(r["classification"])
        old_rank = SEVERITY.index(prior["classification"])

        if new_rank > old_rank:
            cursor.execute("""
                UPDATE data_classification_registry
                SET classification = %s, pii_type = %s, notes = %s,
                    classified_by = 'auto-scanner', classified_at = CURRENT_TIMESTAMP
                WHERE table_name = %s AND column_name = %s
            """, (r["classification"], r["pii_type"],
                  f"Upgraded from {prior['classification']} by content scan",
                  r["table"], r["column"]))
            upgraded += 1
        else:
            kept += 1
            if new_rank < old_rank:
                blocked.append((key, r["classification"], prior))

    conn.commit()
    conn.close()
    return {"inserted": inserted, "upgraded": upgraded, "kept": kept, "blocked": blocked}

def cache_classifications_in_redis():
    """Cache the *registry* in Redis for fast lookups.

    Deliberately sourced from the registry and not from `scan_results`: the
    access-control helper in Step 6 reads this cache, and a cache built from the
    raw scan could hand out a laxer level than the registry actually holds.
    """
    registry = load_registry()
    r = get_redis_client()
    pipe = r.pipeline()

    for (table, column), row in registry.items():
        key = f"classification:{table}:{column}"
        pipe.hset(key, mapping={
            "classification": row["classification"],
            "pii_type": row["pii_type"] or "",
            "classified_by": row["classified_by"] or "",
            "scanned_at": datetime.now().isoformat()
        })
        # Classifications don't change often — cache for 24 hours
        pipe.expire(key, 86400)

    pipe.execute()
    return len(registry)

merge = save_classifications_to_db(scan_results)
cached = cache_classifications_in_redis()

print("✅ Registry merge complete")
print(f"   ➕ inserted {merge['inserted']} new classifications")
print(f"   ⬆️  upgraded {merge['upgraded']} to a more severe level")
print(f"   ⏸️  kept {merge['kept']} existing rows unchanged")
print(f"✅ Cached {cached} registry rows in Redis (24h TTL)")

print("\n🛑 Downgrades refused (the more severe judgement wins):")
for (table, column), scanner_says, prior in merge["blocked"]:
    print(f"   {table}.{column}: scanner said {scanner_says}, "
          f"registry keeps {prior['classification']} (by {prior['classified_by']})")
if not merge["blocked"]:
    print("   (none this run)")

# Demonstrate a Redis lookup
r = get_redis_client()
lookup = r.hgetall("classification:users:ssn")
print(f"\n🔍 Redis lookup for users.ssn: {lookup}")

registry_after = load_registry()
assert registry_after[("users", "ssn")]["classification"] == "restricted"
assert registry_after[("payment_methods", "card_last_four")]["classification"] == "confidential", (
    "the human classification of card_last_four must survive a scanner run"
)
assert lookup.get("classification") == registry_after[("users", "ssn")]["classification"], (
    "Redis cache and registry disagree — the access-control helper reads the cache"
)
print("✅ Registry assertions passed")


## 🛡️ Step 6: Classification-Aware Query Helper

Now that we have a registry, let's build a helper that **checks classifications before returning data**. This is how access control works in practice — your API layer consults the registry and masks or blocks restricted columns.

For example, a support agent should see the customer's name but NOT their SSN.

In [ ]:
# Access levels for different roles
ROLE_ACCESS = {
    "public_api":     ["public"],
    "support_agent":  ["public", "internal", "confidential"],
    "data_engineer":  ["public", "internal"],
    "privacy_officer": ["public", "internal", "confidential", "restricted"],
}

def get_column_classification(table, column):
    """Look up a column's classification, checking Redis first."""
    r = get_redis_client()
    key = f"classification:{table}:{column}"
    cached = r.hgetall(key)

    if cached:
        return cached.get("classification", "restricted")

    # Fallback: unclassified data is treated as RESTRICTED (safe default).
    # This also means an expired or flushed cache fails *closed* — everything
    # gets masked — which is the correct direction for a privacy control.
    return "restricted"

def query_with_access_control(table, columns, role):
    """Query a table but mask columns the role isn't allowed to see."""
    allowed_levels = ROLE_ACCESS.get(role, ["public"])

    # Build column list, replacing restricted columns with masked values
    select_parts = []
    masked_columns = []

    for col in columns:
        classification = get_column_classification(table, col)
        if classification in allowed_levels:
            select_parts.append(f'"{col}"')
        else:
            select_parts.append(f"'[MASKED]' AS \"{col}\"")
            masked_columns.append((col, classification))

    query = f"SELECT {', '.join(select_parts)} FROM \"{table}\" LIMIT 5"

    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()

    return rows, masked_columns

# Demo: Same query, different roles see different data
columns = ["first_name", "last_name", "email", "ssn", "city", "account_status"]

for role in ["public_api", "support_agent", "privacy_officer"]:
    print(f"\n👤 Role: {role}")
    print(f"   Access: {ROLE_ACCESS[role]}")
    print("-" * 80)

    rows, masked = query_with_access_control("users", columns, role)

    # Print header
    header = f"  {'first_name':<12} {'last_name':<12} {'email':<25} {'ssn':<13} {'city':<12} {'status'}"
    print(header)

    for row in rows:
        vals = [str(v)[:12] if v else "" for v in row[:2]]
        vals.append(str(row[2])[:24] if row[2] else "")
        vals.append(str(row[3])[:12] if row[3] else "")
        vals.append(str(row[4])[:12] if row[4] else "")
        vals.append(str(row[5]) if row[5] else "")
        print(f"  {vals[0]:<12} {vals[1]:<12} {vals[2]:<25} {vals[3]:<13} {vals[4]:<12} {vals[5]}")

    if masked:
        print(f"  🔒 Masked: {', '.join(f'{c} ({l})' for c, l in masked)}")

# ── What this control does and does not do ──────────────────────────────────
# Masking happens in the SELECT list: the row is still there, the SSN is still
# in the table, and a database dump, a replica, a backup or a `psql` session
# still hands it over in full. Masking is an *access* control, not deletion.
# Notebook 4 covers the difference between hiding a value and destroying it.
print("\n" + "=" * 80)
print("⚠️  Masking ≠ deletion. '[MASKED]' is applied in the query, not the table.")
print("   The SSN is still on disk, still in backups, still in the WAL.")
print("   Masking also does not stop re-identification: the unmasked columns")
print("   (city, status, signup source) can still single a person out — that is")
print("   what notebook 3's k-anonymity section is about.")

# The controls this notebook claims must actually hold.
assert get_column_classification("users", "column_that_was_never_classified") == "restricted", (
    "unclassified data must default to RESTRICTED"
)
_, masked_for_support = query_with_access_control("users", ["email", "ssn"], "support_agent")
assert [c for c, _ in masked_for_support] == ["ssn"], (
    "a support agent must see the email (confidential) and never the SSN (restricted)"
)
_, masked_for_officer = query_with_access_control("users", ["email", "ssn"], "privacy_officer")
assert masked_for_officer == [], "the privacy officer role sees restricted columns"
_, masked_for_engineer = query_with_access_control("users", ["email", "ssn"], "data_engineer")
assert len(masked_for_engineer) == 2, "a data engineer gets neither confidential nor restricted"
print("\n✅ Access-control assertions passed")


## 🎯 Key Takeaways

1. **PII is everywhere** — it's in obvious places (email column) and non-obvious places (free-text support tickets)
2. **Classification must be automated** — manual review doesn't scale to thousands of tables
3. **Two scanning approaches** — column name patterns catch structured PII, content scanning catches hidden PII
4. **Registry is the source of truth** — every team should be able to look up any column's classification
5. **Default to Restricted** — if you don't know a column's classification, assume the worst
6. **Access control uses classification** — different roles see different columns based on their classification
7. **A clean scan is not a clean database** — our recall harness found 31% of the PII a human found. "No findings" means "no pattern matched", and any process that treats it as proof gets a false sense of safety. Automated scanning narrows where humans have to look; it does not replace them.
8. **Never let a scanner downgrade a human** — the merge in Step 5 only inserts and upgrades. One blind `ON CONFLICT DO UPDATE` and every careful judgement in the registry is gone.
9. **Masking is not deletion** — `[MASKED]` is applied in the SELECT list. The value is still in the table, the backups and the replicas. Notebook 4 covers actual erasure.

### What Microsoft Does

- **Microsoft Purview** scans databases, files, and APIs automatically to classify data (with ML classifiers, not the regexes we used — and it still reports confidence levels rather than certainty)
- Every Azure service must register its data assets in the classification catalog
- Unclassified data in production triggers an alert to the privacy team
- Classification labels flow through to access policies, encryption rules, and retention schedules

### Next Notebook

In **Notebook 2: Privacy Impact Assessment**, we'll use these classifications to evaluate the risk of a new feature before it ships.